# Notebook 03 — Weather-Aware U-Net (Multimodal) · v2.1
**Model B: Multi-scale FiLM U-Net with CBAM attention + Gradient Scaling**

Key upgrades over v2:
- **Gradient Multiplier (2.0x)** applied to weather features to ensure the MLP doesn't get drowned out by image gradients
- Simplified WeatherMLP to prevent feature dominance
- Higher patience (25) and longer warmup (10) for multimodal convergence

In [ ]:
import sys, os
os.chdir(r'd:\flood_segmentation')
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    os.chdir('/content/drive/MyDrive/flood_segmentation')
    get_ipython().run_line_magic('pip', 'install -q rasterio tqdm scikit-learn scipy')

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

sys.path.insert(0, os.path.abspath('.'))
from src.datasets import build_datasets, WEATHER_FEATURES
from src.models   import get_model
from src.train    import train_model, evaluate_model
from src.utils    import visualize_predictions, plot_training_history

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU   : {torch.cuda.get_device_name(0)}')
    print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ── Hyperparameters ─────────────────────────────────────────────────────────
IMG_SIZE    = 256
BATCH_SIZE  = 4     # Physical batch — fits 4GB GPU
ACCUM_STEPS = 2     # Effective batch = 8
NUM_EPOCHS  = 80    # Increased epochs for more stable final refinement
LR          = 3e-4
PATIENCE    = 25    # Vital: allows model through long multimodal plateaus
WARMUP      = 10    # Longer warmup for stable feature fusion
COSINE_T0   = 20
NUM_WORKERS = 0
N_WEATHER   = len(WEATHER_FEATURES)   # 5 features

DATA_DIR    = 'data/raw'
WEATHER_CSV = 'data/processed/weather.csv'
SAVE_PATH   = 'results/saved_models/multimodal_unet_v2.pth'
FIG_DIR     = 'results/figures'
os.makedirs(FIG_DIR, exist_ok=True)
print(f'Weather features ({N_WEATHER}): {WEATHER_FEATURES}')

In [ ]:
# --- Build datasets (same seed = same split as Notebook 02) ---
train_ds, val_ds, test_ds = build_datasets(
    data_dir=DATA_DIR,
    weather_csv_path=WEATHER_CSV,
    img_size=IMG_SIZE,
    train_ratio=0.70,
    val_ratio=0.15,
    seed=42,   # CRITICAL: must match Notebook 02
)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

# Inspect a weather vector
img, weather, mask = train_ds[0]
print(f'\nWeather vector (normalised): {weather.numpy().round(4)}')
for feat, val in zip(WEATHER_FEATURES, weather.numpy()):
    print(f'  {feat:15s}: {val:.4f}')

In [ ]:
# --- Build Model B: Weather-Aware U-Net v2.1 ---
model_B = get_model('multimodal', img_ch=2, weather_dim=N_WEATHER,
                    base_ch=64, drop_prob=0.1)
n_params = sum(p.numel() for p in model_B.parameters() if p.requires_grad)
print(f'WeatherAwareUNet (v2.1 Refined) — Trainable parameters: {n_params:,}')

In [ ]:
# --- Train ---
history_B, best_iou_B = train_model(
    model=model_B,
    train_dataset=train_ds,
    val_dataset=val_ds,
    is_multimodal=True,
    save_path=SAVE_PATH,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    patience=PATIENCE,
    num_workers=NUM_WORKERS,
    device=device,
    warmup_epochs=WARMUP,
    accum_steps=ACCUM_STEPS,
    cosine_T0=COSINE_T0,
)
print(f'\nFinal Best Val IoU (Weather-Aware U-Net): {best_iou_B:.4f}')

In [ ]:
# --- Training curves ---
fig = plot_training_history(
    history_B,
    save_path=f'{FIG_DIR}/03_multimodal_v2_history.png',
    title='Weather-Aware U-Net v2.1 — Training History',
)
plt.show()

In [ ]:
# --- Evaluate on test set (with TTA) ---
test_metrics_B = evaluate_model(
    model=get_model('multimodal', img_ch=2, weather_dim=N_WEATHER,
                    base_ch=64, drop_prob=0.1),
    test_dataset=test_ds,
    checkpoint_path=SAVE_PATH,
    is_multimodal=True,
    batch_size=BATCH_SIZE,
    device=device,
    use_tta=True,
)

os.makedirs('results', exist_ok=True)
with open('results/metrics_multimodal.json', 'w') as f:
    json.dump(test_metrics_B, f, indent=2)
print('Test metrics saved to results/metrics_multimodal.json')

In [ ]:
# --- Final Head-to-Head Comparison ---
import pandas as pd
with open('results/metrics_baseline.json') as f:
    test_metrics_A = json.load(f)

metrics = ['iou', 'dice', 'precision', 'recall', 'f1', 'loss']
comparison_df = pd.DataFrame({
    'Baseline U-Net v2':      {m: test_metrics_A.get(m, float('nan')) for m in metrics},
    'Weather-Aware U-Net (Refined)':  {m: test_metrics_B.get(m, float('nan')) for m in metrics},
}).T

print('\n── FINAL MULTIMODAL COMPARISON ──────────────────────────────')
print(comparison_df.round(4).to_string())
delta_iou = test_metrics_B['iou'] - test_metrics_A['iou']
print(f'\nΔ IoU (Refined Multimodal − Baseline): {delta_iou:+.4f}')